# glcuda r256 verification — Phase B correctness gate

Focused notebook: does the `gl_gemm_mma_q8_r256` kernel (Phase B, 256-row weight reuse) compute the RIGHT answer, and is it actually faster once correct?

Fixes the stale-checkout trap the old notebook hit: `git pull` was silently resolving to the wrong remote and reporting *"Already up to date"* while behind. This fetches the GitHub URL **directly** and hard-resets to it, then proves the fix is on disk before building.

**Runtime:** GPU **T4** + Internet on. First run ~5 min (build + no model download needed — the bench uses synthetic buffers).

Read the cells top to bottom; each prints ONE clear result.

## 1 · GPU + Rust toolchain

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader
import os, subprocess
if os.path.exists('/kaggle') and not os.path.exists('/content'):
    subprocess.run(['ln','-sfn','/kaggle/working','/content'], check=True)
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version

## 2 · Clone + FORCE to the latest commit

`git fetch <URL> main` + `reset --hard FETCH_HEAD` bypasses whatever misconfigured remote made the old notebook say "Already up to date" while stale. The printed commit hash is the ground truth for what's about to build.

In [ ]:
REPO = 'https://github.com/JinXSuperSolo/gwenland-ai'
%cd /content
if not os.path.exists('/content/gwenland-ai'):
    !git clone -q $REPO
%cd /content/gwenland-ai
# Force to the exact remote tip, discarding any local dirt from prior runs.
!git fetch -q $REPO main
!git reset --hard FETCH_HEAD
!git clean -fd glcuda/examples >/dev/null 2>&1
print('\n>>> HEAD is now:')
!git log -1 --oneline

## 3 · Prove the fix is physically on disk

The r256 register-alias fix moves the staging byte offset to `%r45` (four staging passes). If this prints **4**, the corrected kernel is on disk. If it prints **0**, the checkout is still stale — stop and re-run cell 2.

In [ ]:
n = subprocess.run(
    ['grep', '-c', 'add.s32 %r43, %r43, %r45', 'glcuda/src/kernels/glcuda_sm75.ptx'],
    capture_output=True, text=True).stdout.strip()
print(f'staging passes using %r45 (the fix): {n}')
print('OK — corrected kernel on disk' if n == '4' else 'STALE — re-run cell 2, checkout did not advance')

## 4 · Build the bench (watch for `Compiling glcuda`)

If glcuda actually recompiles you'll see a `Compiling glcuda` line and a multi-second build. A `Finished ... in 0.2s` with no compile means a cached (stale) binary — but after the hard reset in cell 2 the PTX changed, so it will rebuild.

In [ ]:
!cargo build --release -p glcuda --example bench 2>&1 | grep -E 'Compiling glcuda|Finished|error' | tail -5

## 5 · ⭐ THE GATE — r256 correctness vs the trusted 8-tile kernel

Runs the SAME inputs through r256 and the shipped, correct `gemm_mma_q8`, then diffs element-wise. This is the verdict:

- **MATCH** → the register-alias fix worked; r256 is correct. Proceed to the (re-taken) timing in cell 6.
- **MISMATCH** → still a bug; the `first bad (t,r)` coordinates localize it. Do NOT trust any timing.

`CUDA_CACHE_DISABLE=1` forces a real ptxas recompile so the JIT can't serve a stale cubin either.

In [ ]:
benv = {**os.environ, 'GLCUDA_JIT_VERBOSE': '1', 'CUDA_CACHE_DISABLE': '1'}
benv.pop('GLCUDA_NO_MMA', None)
r = subprocess.run(['target/release/examples/bench'], capture_output=True, text=True, env=benv)
print('=== r256 correctness ===')
for line in r.stdout.splitlines():
    if line.startswith('[r256-parity'):
        print(line)
if '[r256-parity' not in r.stdout:
    print('no [r256-parity] line — build/run failed; stderr tail:')
    print(r.stderr[-1500:])
# Stash full output for the next cells so we don't re-run the bench.
BENCH_OUT, BENCH_ERR = r.stdout, r.stderr

In [ ]:
# Bug localizer: the [r256-ladder] sweeps shapes from minimal (32 K, 8 rows)
# upward. The FIRST MISMATCH says which dimension triggers it:
#   fails at ntok=8, in=32     -> m-tile 0 / MMA / epilogue (the core)
#   passes 8 but fails 16/64   -> m-tile advance
#   passes small in but fails in>32 -> k-loop accumulation
#   passes <=64 but fails 72/256    -> the 4-pass staging (>64 rows)
print('=== r256 bug localizer (first MISMATCH points at the broken stage) ===')
for line in BENCH_OUT.splitlines():
    if line.startswith('[r256-ladder]') or line.strip().startswith('in='):
        print(line)

## 6 · Timing — ONLY meaningful if cell 5 said MATCH

The [gemm-phaseb] A/B (8×64 vs 4×128 vs 2×256 for a 512-row chunk) and the per-token reuse trend. If cell 5 was MISMATCH these numbers are timing garbage (a wrong kernel doing 1/500th the work is "fast"). Re-taken here on the corrected kernel.

In [ ]:
print('=== timing (trust only if cell 5 = MATCH) ===')
for line in BENCH_OUT.splitlines():
    if line.startswith('[gemm-phaseb') or line.startswith('[gemm-reuse') or line.startswith('[bandwidth]'):
        print(line)

## 7 · r256 register/smem (occupancy sanity)

Confirms the corrected kernel still fits: expect ~96 registers, 9216 B smem, **0 spill**. A spill here would explain a slowdown.

In [ ]:
print('=== r256 register/smem ===')
grab = False
for line in BENCH_ERR.splitlines():
    s = line.strip()
    if 'gl_gemm_mma_q8_r256' in s:
        grab = True
    if grab and ('registers' in s or 'spill' in s or 'smem' in s or 'stack frame' in s):
        print(s)
    if grab and 'Compile time' in s:
        break